# Lab Week 2 · K-means and clustering

Run cells in order with **Shift + Enter**. Read the comments, inspect outputs and write your explanations. Your code runs in this browser. Download your notebook or use the lesson’s PDF button to keep your work; submission is on your course platform.

In [ ]:
# Load the scientific libraries into this browser's Python environment.
import piplite
await piplite.install(["numpy", "pandas", "matplotlib", "scikit-learn"])
print("Libraries ready. Run the dataset cell next.")

## Task 1 · Create and inspect the data

Run the dataset cell below. Identify the observations, features and units. Explain why we scale these features and why the model receives no known customer-group labels.

In [ ]:
# Import the tools used throughout this lab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

# Create 120 synthetic customers. No customer-group labels are supplied to models.
# A fixed random seed makes the example reproducible.
rng = np.random.default_rng(17)
income = np.concatenate([rng.normal(250000, 45000, 40),
                         rng.normal(600000, 70000, 40),
                         rng.normal(950000, 85000, 40)])
spending = np.concatenate([rng.normal(25, 9, 40),
                           rng.normal(75, 9, 40),
                           rng.normal(40, 9, 40)])
df = pd.DataFrame({"annual_income": income, "spending_score": spending})
print("Data shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print(df.head())

# Income and spending use different units. Scale each input feature separately.
# Keep df in original units; X is the numerical matrix used for clustering.
scaler = StandardScaler()
X = scaler.fit_transform(df)
plt.scatter(df["annual_income"], df["spending_score"], color="slategray")
plt.xlabel("Annual income (Rs)")
plt.ylabel("Spending score")
plt.title("Synthetic customers before clustering")
plt.tight_layout()
plt.show()

**My explanation:**

Write your reasoning and interpretation here, or add explanatory comments to your code.

## Task 2 · Fit K-means and inspect the groups

Run K-means with k = 3. Describe what the assignments, centroids and SSE tell you. Explain why the cluster numbers are not verified customer categories.

In [ ]:
# k is a choice made by us, not a known output supplied with each observation.
model = KMeans(n_clusters=3, init="k-means++", n_init=20, random_state=17)
labels = model.fit_predict(X)
print("SSE on standardised features:", round(model.inertia_, 3))
print("Observations per discovered group:")
print(pd.Series(labels).value_counts().sort_index())

# Plot the same feature space used to compute distances and SSE.
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", s=25)
plt.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
            marker="X", s=180, c="crimson", label="Centroids")
plt.xlabel("Standardised annual income")
plt.ylabel("Standardised spending score")
plt.title("K-means: k = 3")
plt.legend()
plt.tight_layout()
plt.show()

**My explanation:**

Write your reasoning and interpretation here, or add explanatory comments to your code.

## Task 3 · Compare different starting centres

Keep X and k fixed. Compare three random initialisations. Explain whether the SSE values or group boundaries differ. Identical results are also a valid finding; explain what you observed.

In [ ]:
# Use one RANDOM start per run so that initialisation differences are visible.
# This experiment deliberately differs from n_init=20 in the previous cell.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharex=True, sharey=True)
start_results = []
for ax, seed in zip(axes, [1, 7, 21]):
    trial = KMeans(n_clusters=3, init="random", n_init=1, random_state=seed)
    trial_labels = trial.fit_predict(X)
    start_results.append({"seed": seed, "SSE": trial.inertia_})
    ax.scatter(X[:, 0], X[:, 1], c=trial_labels, cmap="viridis", s=18)
    ax.scatter(trial.cluster_centers_[:, 0], trial.cluster_centers_[:, 1],
               marker="X", s=100, c="crimson")
    ax.set_title(f"Seed {seed}: SSE {trial.inertia_:.2f}")
    ax.set_xlabel("Standardised income")
axes[0].set_ylabel("Standardised spending")
plt.tight_layout()
plt.show()
print(pd.DataFrame(start_results))
# Cluster numbers can swap between runs; compare geometry and SSE, not colour alone.

**My explanation:**

Write your reasoning and interpretation here, or add explanatory comments to your code.

## Task 4 · Draw an elbow plot

Compare k = 1 to 8 using the same X. Identify where extra clusters bring smaller improvements, if such a bend is visible. Explain why the lowest SSE alone would keep favouring larger k.

In [ ]:
# Refit for each k using multiple initialisations on the SAME prepared data.
fits = {}
rows = []
for k in range(1, 9):
    fit = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=17).fit(X)
    fits[k] = fit
    rows.append({"k": k, "SSE": fit.inertia_})
results = pd.DataFrame(rows)
print("SSE by number of clusters:\n", results.round(3))
plt.plot(results["k"], results["SSE"], "o-", color="teal")
plt.xticks(range(1, 9))
plt.xlabel("Number of clusters, k")
plt.ylabel("SSE (standardised feature space)")
plt.title("Elbow method: look for diminishing improvements")
plt.tight_layout()
plt.show()
# An elbow is a judgement, not a guaranteed unique answer. Repeated fitting
# reduces sensitivity to initialisation but does not guarantee a global minimum.

**My explanation:**

Write your reasoning and interpretation here, or add explanatory comments to your code.

## Task 5 · Compare silhouette scores and justify k

Use silhouette scores alongside the elbow and plots. State a defensible k and explain why. A larger silhouette score is evidence of compact, separated groups under this distance measure, not proof of meaningful real-world classes.

In [ ]:
# Silhouette compares each observation's own-group distance with the
# nearest OTHER group's average distance. It is undefined for k = 1.
scores = {k: silhouette_score(X, fits[k].labels_) for k in range(2, 9)}
results["silhouette"] = results["k"].map(scores)
print("Compare SSE and silhouette (NaN means undefined):")
print(results.round(3))
plt.plot(list(scores), list(scores.values()), "o-", color="purple")
plt.xticks(range(2, 9))
plt.ylim(-1, 1)
plt.xlabel("Number of clusters, k")
plt.ylabel("Mean silhouette score")
plt.title("Silhouette: larger values mean better separation on these features")
plt.tight_layout()
plt.show()
# The silhouette mean lies between -1 and 1. Combine it with your elbow
# interpretation and domain reasoning rather than choosing blindly.

**My explanation:**

Write your reasoning and interpretation here, or add explanatory comments to your code.

## Task 6 · Explore overlapping groups with a GMM

This separate example has overlapping, elongated point clouds. Compare hard K-means assignments with GMM membership probabilities. Inspect the five most ambiguous observations. Explain why those probabilities depend on the fitted model.

In [ ]:
# Create a separate two-feature example to study overlap.
rng_overlap = np.random.default_rng(24)
overlap = np.vstack([
    rng_overlap.multivariate_normal([-1, -0.1], [[1.8, 0.8], [0.8, 0.65]], 70),
    rng_overlap.multivariate_normal([1.25, 0.45], [[0.65, -0.55], [-0.55, 1.6]], 70)
])
Z = StandardScaler().fit_transform(overlap)
hard = KMeans(n_clusters=2, n_init=20, random_state=24).fit_predict(Z)
gmm = GaussianMixture(n_components=2, covariance_type="full", n_init=10,
                     random_state=24).fit(Z)
probabilities = gmm.predict_proba(Z)
print("GMM converged:", gmm.converged_)
print("First five probability rows:\n", probabilities[:5].round(3))

# A hard assignment chooses one group. The GMM can express uncertain membership.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axes[0].scatter(Z[:, 0], Z[:, 1], c=hard, cmap="coolwarm", s=22)
axes[0].set_title("K-means: one group per observation")
plot = axes[1].scatter(Z[:, 0], Z[:, 1], c=probabilities[:, 0],
                       cmap="coolwarm", vmin=0, vmax=1, s=22)
axes[1].set_title("GMM: probability of component 0")
fig.colorbar(plot, ax=axes[1], label="Membership probability")
for ax in axes:
    ax.set_xlabel("Standardised feature 1")
axes[0].set_ylabel("Standardised feature 2")
plt.tight_layout()
plt.show()

# Select observations closest to a 50/50 split between the two components.
ambiguous = np.argsort(np.abs(probabilities[:, 0] - 0.5))[:5]
print("Most ambiguous observations:")
print(pd.DataFrame(probabilities[ambiguous], index=ambiguous,
                   columns=["P(component 0)", "P(component 1)"]).round(3))
# Component numbers are arbitrary. These probabilities are NOT verified class
# probabilities, and a GMM is not guaranteed to outperform K-means.

**My explanation:**

Write your reasoning and interpretation here, or add explanatory comments to your code.

## Task 7 · Your turn: investigate a new dataset

Inspect this new synthetic dataset, prepare its features, fit K-means and compare at least three values of k. Produce elbow and silhouette plots, justify a selected k and discuss one limitation. Try different starts and explain your observations. Optionally compare a GMM. Write your own analysis; no worked solution follows.

In [ ]:
# New synthetic measurements for independent practice. No group labels are supplied.
# Run this cell, then add your OWN preprocessing, fitting and evaluation cells.
rng_practice = np.random.default_rng(53)
practice = pd.DataFrame({
    "feature_a": np.concatenate([rng_practice.normal(-2, 1.1, 50),
                                 rng_practice.normal(1, 0.8, 50),
                                 rng_practice.normal(2.5, 1.2, 50)]),
    "feature_b": np.concatenate([rng_practice.normal(2, 0.8, 50),
                                 rng_practice.normal(-1.5, 1.2, 50),
                                 rng_practice.normal(2, 0.6, 50)])
})
print(practice.head())
print("Independent dataset shape:", practice.shape)
# Keep practice unchanged. Decide which features and transformations you will use.
# Add plots and comments explaining your choices. Do not copy a chosen k blindly.

**My explanation:**

Write your reasoning and interpretation here, or add explanatory comments to your code.